# 211 — Trace Cleanup

Safe deletion of investigation trace data stored in Neo4j.

**Only trace subgraph nodes are ever deleted:**
- `InvestigationTrace` nodes
- `TraceEvent` nodes
- `HAS_EVENT`, `NEXT_EVENT`, and `ABOUT` relationships

**Business graph nodes are never touched:**
`Company`, `Individual`, `LegalEntity`, `Address`, `SIC`

In [1]:
import sys
sys.path.insert(0, "..")

import uuid
import pandas as pd
from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.domain.models import EventType, InvestigationTrace, TraceEvent

settings = get_neo4j_settings()
neo4j = Neo4jRepository(**vars(settings))
neo4j.test_connection()

repo = TraceRepository(neo4j)
print("Connected")

Connected


## Preview — trace counts before cleanup

In [2]:
rows = repo.list_traces(limit=100)
total_events = sum(r["event_count"] for r in rows)
print(f"Traces in database : {len(rows)}")
print(f"Total events       : {total_events}")
if rows:
    display(pd.DataFrame(rows)[["trace_id", "query", "mode", "started_at", "event_count"]])

Traces in database : 0
Total events       : 0


## Seed test data

Create three short traces — two for entity A, one for entity B —
so each delete method has something to work on.

In [3]:
ENTITY_A = "ACME HOLDINGS LIMITED"
ENTITY_B = "BETA VENTURES LTD"


def _make_trace(entity: str, mode: str = "interactive") -> InvestigationTrace:
    t = InvestigationTrace(
        request_id=str(uuid.uuid4()),
        entity_name=entity,
        user_id="analyst-211",
        mode=mode,
    )
    repo.save_trace(t)
    repo.append_event(
        t.request_id,
        TraceEvent(
            event_type=EventType.TOOL_RETURNED,
            message="ownership_complexity_check returned LOW risk.",
            payload={
                "tool_name": "ownership_complexity_check",
                "output_summary": "risk_level=LOW, max_depth=1",
                "decision": "no escalation needed",
            },
        ),
    )
    repo.finalize_trace(t.request_id, final_summary=f"Clean investigation of {entity}.")
    return t


trace_a1 = _make_trace(ENTITY_A)          # first trace for entity A
trace_a2 = _make_trace(ENTITY_A)          # second trace for entity A
trace_b  = _make_trace(ENTITY_B)          # single trace for entity B

print(f"trace_a1 : {trace_a1.request_id}  ({ENTITY_A})")
print(f"trace_a2 : {trace_a2.request_id}  ({ENTITY_A})")
print(f"trace_b  : {trace_b.request_id}  ({ENTITY_B})")

trace_a1 : 2e9dda54-fda8-4539-a306-9c0346730617  (ACME HOLDINGS LIMITED)
trace_a2 : 5dcf52f3-2692-4091-9543-22c7f1ce72b9  (ACME HOLDINGS LIMITED)
trace_b  : 9d95926f-3901-4904-a32a-6faac32d72d8  (BETA VENTURES LTD)


In [4]:
rows = repo.list_traces(limit=100)
print(f"Traces after seeding: {len(rows)}")
display(pd.DataFrame(rows)[["trace_id", "query", "started_at", "event_count"]])

Traces after seeding: 3


,trace_id,query,started_at,event_count
0,9d95926f-3901-4904-a32a-6faac32d72d8,BETA VENTURES LTD,2026-03-22T05:24:24.723457+00:00,1
1,5dcf52f3-2692-4091-9543-22c7f1ce72b9,ACME HOLDINGS LIMITED,2026-03-22T05:24:24.651603+00:00,1
2,2e9dda54-fda8-4539-a306-9c0346730617,ACME HOLDINGS LIMITED,2026-03-22T05:24:24.539261+00:00,1


## Delete a single trace by `trace_id`

Deletes exactly one `InvestigationTrace` node and all its `TraceEvent` nodes.
No business graph nodes are affected.

In [5]:
target_id = trace_a1.request_id
result = repo.delete_trace(target_id)

print(f"trace_id        : {target_id}")
print(f"found           : {result['found']}")
print(f"traces_deleted  : {result['traces_deleted']}")
print(f"events_deleted  : {result['events_deleted']}")

# Miss case — same ID again
miss = repo.delete_trace(target_id)
print(f"\nMiss case (same ID): found={miss['found']}, deleted={miss['traces_deleted']}")

trace_id        : 2e9dda54-fda8-4539-a306-9c0346730617
found           : True
traces_deleted  : 1
events_deleted  : 1

Miss case (same ID): found=False, deleted=0


## Delete traces by entity name

Matches and removes all traces where:
- the trace `query` field equals the entity name, **or**
- any event in the trace has an `:ABOUT` relationship to a node with that name.

> **Note:** Only trace data is deleted. The `Company` (or other business) node
> with that name is **not** affected.

In [6]:
result = repo.delete_traces_by_entity(ENTITY_A)

print(f"entity_name     : {ENTITY_A}")
print(f"found           : {result['found']}")
print(f"traces_deleted  : {result['traces_deleted']}")
print(f"events_deleted  : {result['events_deleted']}")

# Miss case — no more traces for entity A
miss = repo.delete_traces_by_entity(ENTITY_A)
print(f"\nMiss case: found={miss['found']}, deleted={miss['traces_deleted']}")

entity_name     : ACME HOLDINGS LIMITED
found           : True
traces_deleted  : 1
events_deleted  : 1

Miss case: found=False, deleted=0


## Delete ALL traces

> ⚠️ **This wipes the entire trace subgraph — all `InvestigationTrace` and
> `TraceEvent` nodes. This action cannot be undone.**
>
> Business graph nodes (`Company`, `Individual`, `LegalEntity`, `Address`,
> `SIC`) are **never** deleted by this operation. Only the audit trail is
> removed.

In [7]:
result = repo.delete_all_traces()

print(f"traces_deleted  : {result['traces_deleted']}")
print(f"events_deleted  : {result['events_deleted']}")

traces_deleted  : 1
events_deleted  : 1


## Preview — trace counts after cleanup

In [8]:
rows = repo.list_traces(limit=100)
total_events = sum(r["event_count"] for r in rows)
print(f"Traces remaining : {len(rows)}")
print(f"Events remaining : {total_events}")

Traces remaining : 0
Events remaining : 0


In [9]:
neo4j.close()
print("Driver closed")

Driver closed
